# Day 1 - Live NAV Fetch
Bluestock MF Analytics Capstone

Fetches live NAV history from mfapi.in for the schemes named in the capstone brief.

In [ ]:
import requests
import pandas as pd

### Scheme codes from the brief
HDFC Top 100 is the primary task; the other 5 are the required comparison set.

In [ ]:
schemes = {
    125497: "hdfc_top100",
    119551: "sbi_bluechip",
    120503: "icici_bluechip",
    118632: "nippon_large_cap",
    119092: "axis_bluechip",
    120841: "kotak_bluechip",
}
schemes

### Try one scheme first - HDFC Top 100 (125497)

In [ ]:
url = "https://api.mfapi.in/mf/125497"

response = requests.get(url, timeout=15)
response.status_code

In [ ]:
payload = response.json()
payload["meta"]

In [ ]:
records = payload["data"]
len(records)

In [ ]:
hdfc_nav = pd.DataFrame(records)
hdfc_nav.head()

In [ ]:
hdfc_nav["date"] = pd.to_datetime(hdfc_nav["date"], format="%d-%m-%Y")
hdfc_nav["nav"] = hdfc_nav["nav"].astype(float)

hdfc_nav = hdfc_nav.sort_values("date").reset_index(drop=True)
hdfc_nav.head()

In [ ]:
hdfc_nav.to_csv("../data/raw/hdfc_top100_live_nav.csv", index=False)
print("Saved", len(hdfc_nav), "rows")

### Now do the same for all 6 schemes in a loop

In [ ]:
for code_, name in schemes.items():
    print("Fetching:", name, code_)

    url = f"https://api.mfapi.in/mf/{code_}"
    response = requests.get(url, timeout=15)

    if response.status_code != 200:
        print("  failed, status:", response.status_code)
        continue

    payload = response.json()
    records = payload.get("data", [])

    if not records:
        print("  no data returned")
        continue

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")
    df["nav"] = df["nav"].astype(float)
    df = df.sort_values("date").reset_index(drop=True)

    out_path = f"../data/raw/{name}_live_nav.csv"
    df.to_csv(out_path, index=False)

    print("  saved", len(df), "rows ->", out_path)

### Quick check - list the live NAV files we just created

In [ ]:
import os

[f for f in os.listdir("../data/raw") if "live_nav" in f]